In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (confusion_matrix, classification_report,
                              precision_score, recall_score, f1_score, accuracy_score)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('bank.csv', sep=';')
df.head()

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df['y'].value_counts()

In [ ]:
df_clean = df.copy()

cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    df_clean[col] = le.fit_transform(df_clean[col])

df_clean.head()

In [ ]:
X = df_clean.drop('y', axis=1)
y = df_clean['y']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

counts = df['y'].value_counts()
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=['#4e79a7', '#f28e2b'],
            wedgeprops=dict(edgecolor='white', linewidth=2), startangle=90)
axes[0].set_title('Subscription Distribution', fontsize=13, fontweight='bold')

mar = df['marital'].value_counts()
axes[1].pie(mar, labels=mar.index, autopct='%1.1f%%',
            colors=['#59a14f', '#e15759', '#76b7b2'],
            wedgeprops=dict(edgecolor='white', linewidth=2), startangle=90)
axes[1].set_title('Marital Status', fontsize=13, fontweight='bold')

edu = df['education'].value_counts()
axes[2].pie(edu, labels=edu.index, autopct='%1.1f%%',
            colors=['#edc948', '#b07aa1', '#ff9da7', '#9c755f'],
            wedgeprops=dict(edgecolor='white', linewidth=2), startangle=90)
axes[2].set_title('Education Level', fontsize=13, fontweight='bold')

plt.suptitle('Pie Charts', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ['#4e79a7','#f28e2b','#e15759','#76b7b2','#59a14f',
          '#edc948','#b07aa1','#ff9da7','#9c755f','#bab0ac','#4e79a7']

job = df['job'].value_counts()
axes[0].bar(job.index, job.values, color=colors[:len(job)], edgecolor='white')
axes[0].set_title('Job Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Job Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=35)
for i, v in enumerate(job.values):
    axes[0].text(i, v + 15, str(v), ha='center', fontsize=8)

ed_sub = df.groupby('education')['y'].value_counts().unstack().fillna(0)
ed_sub.plot(kind='bar', ax=axes[1], color=['#4e79a7', '#f28e2b'],
            edgecolor='white', legend=True)
axes[1].set_title('Subscription by Education', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Education')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Bar Charts', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

month_order = ['jan','feb','mar','apr','may','jun',
               'jul','aug','sep','oct','nov','dec']
month_counts = df['month'].value_counts().reindex(month_order).dropna()

axes[0].plot(month_counts.index, month_counts.values,
             marker='o', color='#4e79a7', linewidth=2.5,
             markersize=8, markerfacecolor='#f28e2b')
axes[0].fill_between(range(len(month_counts)), month_counts.values,
                     alpha=0.15, color='#4e79a7')
axes[0].set_xticks(range(len(month_counts)))
axes[0].set_xticklabels(month_counts.index, rotation=30)
axes[0].set_title('Calls per Month', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Number of Calls')
axes[0].grid(True, alpha=0.3)

df['age_group'] = pd.cut(df['age'], bins=[18, 30, 40, 50, 60, 100],
                          labels=['18-30', '31-40', '41-50', '51-60', '60+'])
avg_bal = df.groupby('age_group', observed=True)['balance'].mean()
axes[1].plot(avg_bal.index.astype(str), avg_bal.values,
             marker='s', color='#59a14f', linewidth=2.5,
             markersize=9, markerfacecolor='#e15759')
axes[1].set_title('Avg Balance by Age Group', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Average Balance')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Line Charts', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)

print("Training Results")
print("="*50)
print(classification_report(y_train, y_train_pred, target_names=['No', 'Yes']))

In [ ]:
cm_train = confusion_matrix(y_train, y_train_pred)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm_train, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted No', 'Predicted Yes'])
ax.set_yticklabels(['Actual No', 'Actual Yes'])
ax.set_title('Confusion Matrix - Training Data', fontweight='bold')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_train[i, j]), ha='center', va='center',
                fontsize=18, fontweight='bold',
                color='white' if cm_train[i, j] > cm_train.max()/2 else 'black')
plt.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
train_acc  = round(accuracy_score(y_train, y_train_pred)  * 100, 2)
train_prec = round(precision_score(y_train, y_train_pred) * 100, 2)
train_rec  = round(recall_score(y_train, y_train_pred)    * 100, 2)
train_f1   = round(f1_score(y_train, y_train_pred)        * 100, 2)

print("Training Metrics")
print("-" * 35)
print(f"  Accuracy  : {train_acc}%")
print(f"  Precision : {train_prec}%")
print(f"  Recall    : {train_rec}%")
print(f"  F1-Score  : {train_f1}%")

In [ ]:
y_test_pred = model.predict(X_test)

print("Testing Results")
print("="*50)
print(classification_report(y_test, y_test_pred, target_names=['No', 'Yes']))

In [ ]:
cm_test = confusion_matrix(y_test, y_test_pred)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm_test, cmap='Oranges')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted No', 'Predicted Yes'])
ax.set_yticklabels(['Actual No', 'Actual Yes'])
ax.set_title('Confusion Matrix - Testing Data', fontweight='bold')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_test[i, j]), ha='center', va='center',
                fontsize=18, fontweight='bold',
                color='white' if cm_test[i, j] > cm_test.max()/2 else 'black')
plt.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
test_acc  = round(accuracy_score(y_test, y_test_pred)  * 100, 2)
test_prec = round(precision_score(y_test, y_test_pred) * 100, 2)
test_rec  = round(recall_score(y_test, y_test_pred)    * 100, 2)
test_f1   = round(f1_score(y_test, y_test_pred)        * 100, 2)

results = pd.DataFrame({
    'Dataset'  : ['Training', 'Testing'],
    'Accuracy' : [f'{train_acc}%',  f'{test_acc}%'],
    'Precision': [f'{train_prec}%', f'{test_prec}%'],
    'Recall'   : [f'{train_rec}%',  f'{test_rec}%'],
    'F1-Score' : [f'{train_f1}%',   f'{test_f1}%'],
})

results.set_index('Dataset', inplace=True)
results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.5))
ax.axis('off')

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
train_vals = [train_acc, train_prec, train_rec, train_f1]
test_vals  = [test_acc,  test_prec,  test_rec,  test_f1]

cell_data = [
    [f'{v}%' for v in train_vals],
    [f'{v}%' for v in test_vals],
]

tbl = ax.table(cellText=cell_data, rowLabels=['Training', 'Testing'],
               colLabels=metrics, cellLoc='center', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(12)
tbl.scale(1.5, 2.2)

for j in range(len(metrics)):
    tbl[(0, j)].set_facecolor('#4e79a7')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')
tbl[(1, -1)].set_facecolor('#d0e4f7'); tbl[(1, -1)].set_text_props(fontweight='bold')
tbl[(2, -1)].set_facecolor('#cce5cc'); tbl[(2, -1)].set_text_props(fontweight='bold')

ax.set_title('Model Performance Summary', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()